# LangChain 실습 노트북 세팅 및 테스트

## 테스트 범위

- OpenRouter 기반 LangChain 채팅 모델 호출
- 기본 체인 구성: `ChatPromptTemplate | 모델 | StrOutputParser()`
- 로컬 GPU 임베딩 기반 RAG

## 모델 설정

- 기본 라우터: `openai/gpt-oss-20b:free`
- 기본 환경 변수: `OPENROUTER_MODEL`
- RAG 임베딩: `HuggingFaceEmbeddings`
- RAG 디바이스: `cuda`


## 참고 자료

- LangChain `ChatOpenAI`: https://docs.langchain.com/oss/python/integrations/chat/openai
- LangChain Structured Output: https://python.langchain.com/docs/how_to/structured_output/
- LangChain Sentence Transformers: https://docs.langchain.com/oss/python/integrations/embeddings/sentence_transformers
- OpenRouter Quickstart: https://openrouter.ai/docs/quickstart
- OpenRouter API 개요: https://openrouter.ai/docs/api/reference/overview


## 0. 설치와 실행 준비

- 설치:

```bash
uv sync
```

- 필수 환경 변수: `OPENROUTER_API_KEY`
- 선택 환경 변수: `OPENROUTER_BASE_URL`
- 선택 환경 변수: `OPENROUTER_MODEL`
- 설정 파일 로드 순서: `.env-example`
- 설정 파일 로드 순서: `.env`
- 설정 파일 로드 순서: OS 환경 변수
- 기본값: `OPENROUTER_BASE_URL=https://openrouter.ai/api/v1`
- 기본값: `OPENROUTER_MODEL=openai/gpt-oss-20b:free`
- RAG 전제: `langchain-huggingface`
- RAG 전제: `sentence-transformers`
- RAG 전제: CUDA 사용 가능 GPU
- RAG 전제: CUDA 지원 PyTorch
- 주의: 무료 엔드포인트는 입력이 로깅될 수 있음
- 주의: 민감한 데이터 입력 금지


In [1]:
import json
import re
import sys
import textwrap
from pathlib import Path

import torch
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

project_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "langchain_cookbook" / "openrouter_setup.py").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("src/langchain_cookbook/openrouter_setup.py 파일을 찾지 못했습니다.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from langchain_cookbook.openrouter_setup import load_openrouter_settings, make_chat_model

settings = load_openrouter_settings()
OPENROUTER_BASE_URL = settings.base_url
CHAT_MODEL = settings.model


In [2]:
llm = make_chat_model()

print(f"chat 모델: {CHAT_MODEL}")
print(f"base URL: {OPENROUTER_BASE_URL}")


chat 모델: openai/gpt-oss-20b:free
base URL: https://openrouter.ai/api/v1


## 1. 단일 모델 호출

- 목표:
  - `ChatOpenAI` 객체 생성
  - `invoke()` 호출 결과 확인
- 확인 항목:
  - 응답 객체 생성 여부
  - `content` 출력 형식


In [3]:
first_response = llm.invoke(
    "LangChain이 무엇인지 한국어로 3문장만 사용해서 아주 쉽게 설명해줘."
)

print(first_response.content)


LangChain은 인공지능이 자연어를 이해하고, 질문에 답하거나 글을 쓰는 데 도움을 주는 도구 모음이에요.  
이 도구들을 사용하면 챗봇, 자동 번역기, 문서 요약기 같은 프로그램을 쉽게 만들 수 있죠.  
쉽게 말해, LangChain은 AI를 활용해 다양한 언어 작업을 빠르고 편리하게 해주는 “프로그래밍용 레시피” 같은 거예요.


## 2. 기본 체인 패턴

```python
prompt | model | parser
```

- `prompt`:
  - 입력 구조 정의
  - 시스템 지시와 사용자 입력 분리
- `model`:
  - LLM 호출 실행
- `parser`:
  - 응답을 문자열 또는 구조화 데이터로 변환
- 현재 예제 구성:
  - `ChatPromptTemplate`
  - `llm`
  - `StrOutputParser()`


In [3]:
study_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 랭체인 입문 튜터입니다. 설명은 쉽고 짧게 하고, 마지막 줄에 한 줄 실습 팁을 추가하세요.",
        ),
        (
            "human",
            "주제: {topic}\n학습자 수준: {level}\n위 조건에 맞는 설명을 작성해줘.",
        ),
    ]
)

study_chain = study_prompt | llm | StrOutputParser()

study_result = study_chain.invoke(
    {"topic": "PromptTemplate와 Output Parser", "level": "LangChain을 처음 보는 사람"}
)

print(study_result)


**PromptTemplate**  
- 템플릿 문자열에 변수를 넣어 “{name}님, 오늘 날씨는 {weather}입니다.”처럼 동적으로 문장을 만들 수 있어요.  
- `PromptTemplate.from_template("Hello, {name}!")`처럼 만들고, `format(name="Alice")` 하면 “Hello, Alice!”가 됩니다.  

**Output Parser**  
- 모델이 돌려준 텍스트를 원하는 형식(숫자, JSON, 리스트 등)으로 변환해 주는 도구예요.  
- 예를 들어 `response = "1. Apple\n2. Banana"` 를 `ListOutputParser()` 로 파싱하면 `["Apple", "Banana"]` 가 됩니다.  

**핵심 흐름**  
1. PromptTemplate 으로 입력 문장 만들기  
2. LangChain 에서 모델 실행  
3. Output Parser 로 결과를 구조화  

**실습 팁**  
`PromptTemplate` 와 `OutputParser` 를 한 줄씩 테스트해 보면서, 변수가 바뀔 때마다 결과가 어떻게 달라지는지 눈으로 확인해 보세요.


## 3. 로컬 GPU 임베딩으로 RAG 인덱스 만들기

- 임베딩 구현: `HuggingFaceEmbeddings`
- 임베딩 모델: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- 디바이스: `cuda`
- 벡터 저장소: `InMemoryVectorStore`
- 문서 분할: `RecursiveCharacterTextSplitter`
- 전제: CUDA 사용 가능 GPU
- 전제: CUDA 지원 PyTorch


In [4]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU를 찾지 못했습니다. CUDA 지원 PyTorch를 설치한 뒤 다시 실행하세요."
    )

HF_EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=HF_EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
)

rag_docs = [
    Document(
        page_content=textwrap.dedent(
            """
            RAG는 질문과 관련된 문서를 먼저 검색한 뒤, 검색된 문맥을 모델 입력에 함께 넣어 답변 정확도를 높이는 방식이다.
            검색 단계와 생성 단계를 분리하면 품질 저하 지점을 추적하기 쉽다.
            검색 결과의 source를 함께 보여주면 검증 가능성이 높아진다.
            """
        ).strip(),
        metadata={"source": "rag_overview.md", "topic": "RAG"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            임베딩은 텍스트를 고정 길이 벡터로 바꾸는 과정이다.
            의미가 비슷한 문장은 벡터 공간에서도 가까워지므로 유사도 검색에 사용할 수 있다.
            검색 품질은 임베딩 모델, 청크 크기, overlap 설정의 영향을 받는다.
            """
        ).strip(),
        metadata={"source": "embedding_basics.md", "topic": "Embedding"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            청크 분할은 긴 문서를 검색 가능한 단위로 나누는 과정이다.
            overlap은 문장이 경계에서 끊길 때 문맥 손실을 줄이는 데 도움이 된다.
            너무 작은 청크는 의미가 약해지고, 너무 큰 청크는 검색 정밀도가 떨어질 수 있다.
            """
        ).strip(),
        metadata={"source": "chunking.md", "topic": "Chunking"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            실무 RAG에서는 검색된 문맥 안에서만 답하도록 프롬프트를 제한해야 한다.
            문맥에 없는 내용은 모른다고 답하게 해야 환각을 줄일 수 있다.
            검색 결과와 최종 답변을 함께 로그로 남기면 운영과 디버깅이 쉬워진다.
            """
        ).strip(),
        metadata={"source": "rag_ops.md", "topic": "Operations"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=160, chunk_overlap=30)
rag_splits = splitter.split_documents(rag_docs)

vector_store = InMemoryVectorStore(hf_embeddings)
vector_store.add_documents(rag_splits)

print(f"embedding model: {HF_EMBED_MODEL}")
print("device: cuda")
print(f"원본 문서 수: {len(rag_docs)}")
print(f"분할된 청크 수: {len(rag_splits)}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
device: cuda
원본 문서 수: 4
분할된 청크 수: 4


## 4. 검색 결과로 답변 생성

- 입력: 사용자 질문
- 검색: 상위 3개 청크
- 답변 규칙: 검색 문맥만 사용
- 답변 규칙: 문맥에 없으면 모른다고 답변
- 출력: 답변 + source


In [5]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[source: {doc.metadata['source']}]\\n{doc.page_content}" for doc in docs
    )


retriever = vector_store.as_retriever(search_kwargs={"k": 3})

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "제공된 문맥만 사용해 답하세요. 문맥에 없으면 '모르겠습니다.'라고 답하세요. 마지막 줄에는 참고한 source를 콤마로 정리하세요.",
        ),
        (
            "human",
            "질문: {question}\\n\\n참고 문맥:\\n{context}",
        ),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

question = "RAG에서 source를 함께 보여줘야 하는 이유와, overlap이 필요한 이유를 설명해줘."
retrieved_docs = retriever.invoke(question)
context = format_docs(retrieved_docs)

print("[검색된 문서]")
for idx, doc in enumerate(retrieved_docs, start=1):
    print(f"{idx}. source={doc.metadata['source']}, topic={doc.metadata['topic']}")
    print(doc.page_content)
    print("-" * 80)

print("\n[RAG 답변]")
print(rag_chain.invoke({"question": question, "context": context}))


[검색된 문서]
1. source=rag_overview.md, topic=RAG
RAG는 질문과 관련된 문서를 먼저 검색한 뒤, 검색된 문맥을 모델 입력에 함께 넣어 답변 정확도를 높이는 방식이다.
검색 단계와 생성 단계를 분리하면 품질 저하 지점을 추적하기 쉽다.
검색 결과의 source를 함께 보여주면 검증 가능성이 높아진다.
--------------------------------------------------------------------------------
2. source=chunking.md, topic=Chunking
청크 분할은 긴 문서를 검색 가능한 단위로 나누는 과정이다.
overlap은 문장이 경계에서 끊길 때 문맥 손실을 줄이는 데 도움이 된다.
너무 작은 청크는 의미가 약해지고, 너무 큰 청크는 검색 정밀도가 떨어질 수 있다.
--------------------------------------------------------------------------------
3. source=embedding_basics.md, topic=Embedding
임베딩은 텍스트를 고정 길이 벡터로 바꾸는 과정이다.
의미가 비슷한 문장은 벡터 공간에서도 가까워지므로 유사도 검색에 사용할 수 있다.
검색 품질은 임베딩 모델, 청크 크기, overlap 설정의 영향을 받는다.
--------------------------------------------------------------------------------

[RAG 답변]
RAG에서 **source를 함께 보여줘야 하는 이유**  
- 검색 단계와 생성 단계를 분리하면 품질 저하 지점을 추적하기 쉽다.  
- 검색 결과의 source를 함께 보여주면 검증 가능성이 높아진다.  
  즉, 사용자는 답변이 어디서 온 것인지 바로 확인할 수 있어 신뢰성을 높일 수 있다.  

RAG에서 **overlap이 필요한 이유**  
- 청크 분할 시 문장이 경계에서 끊길 때 문맥 손실을 줄이

## 마무리

- 완료 항목: `ChatOpenAI` 연결
- 완료 항목: 기본 체인 구성
- 완료 항목: `batch()` 처리
- 완료 항목: JSON 후처리
- 완료 항목: 단계 연결
- 완료 항목: `HuggingFaceEmbeddings` + GPU RAG
- 전제: CUDA 사용 가능 GPU
- 전제: CUDA 지원 PyTorch
